# 06 — Offline Model Export, Quantization & Local Inference

This notebook converts the trained Logistic Regression from Notebook 05 into a compact offline artifact and proves that inference can run without scikit-learn or a network call.

Flow:

`trained model → extract parameters → standalone NumPy inference → optional INT8 representation → local rule engine → offline validation`

The model is intentionally tiny. INT8 is an optimization experiment, not a requirement.

Performance here is engineering validation on synthetic data, not clinical or real-world validation.


In [1]:
import os
import json
import time
import joblib
import numpy as np
import pandas as pd

MODEL_PATH = "../models/risk_logistic_regression.joblib"
RULE_PATH = "../models/risk_rule_engine.json"
DATA_PATH = "../data/processed/final_synthetic_risk_dataset.csv"

EXPORT_DIR = "../models/offline"
os.makedirs(EXPORT_DIR, exist_ok=True)

MODEL_FEATURES = [
    "safe_now",
    "perpetrator_present",
    "can_leave_safely",
    "medical_help",
    "contact_requested"
]

print("Offline export stage initialized.")


Offline export stage initialized.


## 1. Load trained model, rules and frozen dataset

In [2]:
ml_pipeline = joblib.load(MODEL_PATH)

with open(RULE_PATH, "r", encoding="utf-8") as f:
    rule_spec = json.load(f)

df = pd.read_csv(DATA_PATH, encoding="latin1")

classifier = ml_pipeline.named_steps["classifier"]
scaler = ml_pipeline.named_steps["scaler"]

print("Dataset:", df.shape)
print("Classes:", classifier.classes_)
print("Features:", MODEL_FEATURES)
print("Rule version:", rule_spec["version"])


Dataset: (20000, 37)
Classes: ['HIGH' 'LOW' 'MEDIUM']
Features: ['safe_now', 'perpetrator_present', 'can_leave_safely', 'medical_help', 'contact_requested']
Rule version: 1.0


## 2. Extract model parameters

In [3]:
means = scaler.mean_.astype(np.float64)
scales = scaler.scale_.astype(np.float64)
coefficients = classifier.coef_.astype(np.float64)
intercepts = classifier.intercept_.astype(np.float64)
classes = classifier.classes_.tolist()

print("Coefficient shape:", coefficients.shape)
print("Intercept shape:", intercepts.shape)
print("Scaler shape:", means.shape)

print("\nMeans:", means)
print("Scales:", scales)
print("Coefficients:\n", coefficients)
print("Intercepts:", intercepts)


Coefficient shape: (3, 5)
Intercept shape: (3,)
Scaler shape: (5,)

Means: [0.86128571 0.11185714 0.83671429 0.05071429 0.14592857]
Scales: [0.34564813 0.31519061 0.36962615 0.21941364 0.35303459]
Coefficients:
 [[-5.06534914  4.45581859 -3.63663422  1.81333847 -0.14029911]
 [ 4.81946739 -4.23579235  4.32797323 -2.41765989 -0.0437342 ]
 [ 0.24588176 -0.22002624 -0.69133902  0.60432142  0.18403331]]
Intercepts: [-17.18749204   8.90109694   8.2863951 ]


## 3. Standalone NumPy inference

This reproduces the Logistic Regression mathematics directly:

`z = ((x - mean) / scale) @ coefficients.T + intercept`

Then softmax converts logits to class probabilities.


In [4]:
def softmax(logits):
    logits = np.asarray(logits, dtype=np.float64)
    shifted = logits - np.max(logits)
    exp_values = np.exp(shifted)
    return exp_values / exp_values.sum()


def standalone_predict_proba(X):
    X = np.asarray(X, dtype=np.float64)

    if X.ndim == 1:
        X = X.reshape(1, -1)

    X_scaled = (X - means) / scales
    logits = X_scaled @ coefficients.T + intercepts

    return np.vstack([
        softmax(row)
        for row in logits
    ])


def standalone_predict(X):
    probabilities = standalone_predict_proba(X)
    indices = np.argmax(probabilities, axis=1)

    return np.array([
        classes[i]
        for i in indices
    ])


## 4. Verify standalone inference against scikit-learn

In [5]:
X_all = df[MODEL_FEATURES].astype(float).values

sklearn_pred = ml_pipeline.predict(
    df[MODEL_FEATURES]
)

standalone_pred = standalone_predict(X_all)

agreement = np.mean(
    sklearn_pred == standalone_pred
)

print("Prediction agreement:", agreement)
print(
    "Mismatches:",
    np.sum(sklearn_pred != standalone_pred)
)


Prediction agreement: 1.0
Mismatches: 0


## 5. Verify probability equivalence

In [6]:
sample_X = X_all[:1000]

sklearn_prob = ml_pipeline.predict_proba(
    sample_X
)

standalone_prob = standalone_predict_proba(
    sample_X
)

print(
    "Maximum probability difference:",
    np.max(np.abs(sklearn_prob - standalone_prob))
)

print(
    "Mean probability difference:",
    np.mean(np.abs(sklearn_prob - standalone_prob))
)


Maximum probability difference: 0.0
Mean probability difference: 0.0


C:\Users\divya\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


## 6. Export floating-point offline model

In [7]:
float_artifact = {
    "version": "1.0",
    "model_type": "multinomial_logistic_regression",
    "features": MODEL_FEATURES,
    "classes": classes,
    "scaler": {
        "mean": means.tolist(),
        "scale": scales.tolist()
    },
    "classifier": {
        "coefficients": coefficients.tolist(),
        "intercepts": intercepts.tolist()
    },
    "deployment": {
        "offline": True,
        "network_required_for_inference": False
    }
}

FLOAT_PATH = f"{EXPORT_DIR}/risk_model_float.json"

with open(FLOAT_PATH, "w", encoding="utf-8") as f:
    json.dump(float_artifact, f, indent=2)

float_size = os.path.getsize(FLOAT_PATH)

print("Saved:", FLOAT_PATH)
print("Size:", float_size, "bytes")


Saved: ../models/offline/risk_model_float.json
Size: 1365 bytes


## 7. Optional INT8 classifier representation

The classifier coefficients and intercepts are quantized to signed INT8 using symmetric per-tensor scaling.

The scaler remains floating point.

This is an engineering experiment to measure whether the smaller representation changes predictions.


In [8]:
max_abs = max(
    np.max(np.abs(coefficients)),
    np.max(np.abs(intercepts))
)

int8_scale = (
    max_abs / 127.0
    if max_abs != 0
    else 1.0
)

coefficients_int8 = np.round(
    coefficients / int8_scale
).clip(-127, 127).astype(np.int8)

intercepts_int8 = np.round(
    intercepts / int8_scale
).clip(-127, 127).astype(np.int8)

coefficients_dequant = (
    coefficients_int8.astype(np.float64)
    * int8_scale
)

intercepts_dequant = (
    intercepts_int8.astype(np.float64)
    * int8_scale
)

print("INT8 scale:", int8_scale)
print("Coefficient dtype:", coefficients_int8.dtype)
print("Intercept dtype:", intercepts_int8.dtype)


INT8 scale: 0.13533458298276893
Coefficient dtype: int8
Intercept dtype: int8


## 8. Test INT8 inference

In [9]:
def standalone_int8_predict_proba(X):
    X = np.asarray(X, dtype=np.float64)

    if X.ndim == 1:
        X = X.reshape(1, -1)

    X_scaled = (X - means) / scales

    logits = (
        X_scaled @ coefficients_dequant.T
        + intercepts_dequant
    )

    return np.vstack([
        softmax(row)
        for row in logits
    ])


def standalone_int8_predict(X):
    probabilities = standalone_int8_predict_proba(X)
    indices = np.argmax(probabilities, axis=1)

    return np.array([
        classes[i]
        for i in indices
    ])


int8_pred = standalone_int8_predict(X_all)

print(
    "INT8 prediction agreement:",
    np.mean(int8_pred == sklearn_pred)
)

print(
    "INT8 mismatches:",
    np.sum(int8_pred != sklearn_pred)
)

print(
    "INT8 max probability difference:",
    np.max(
        np.abs(
            sklearn_prob -
            standalone_int8_predict_proba(sample_X)
        )
    )
)


INT8 prediction agreement: 1.0
INT8 mismatches: 0
INT8 max probability difference: 0.0390026979044098


## 9. Export INT8 artifact

In [10]:
int8_artifact = {
    "version": "1.0",
    "model_type": "multinomial_logistic_regression_int8",
    "features": MODEL_FEATURES,
    "classes": classes,
    "scaler": {
        "mean": means.tolist(),
        "scale": scales.tolist()
    },
    "classifier": {
        "coefficients_int8": coefficients_int8.tolist(),
        "intercepts_int8": intercepts_int8.tolist(),
        "quantization": {
            "scheme": "symmetric_per_tensor",
            "scale": float(int8_scale),
            "zero_point": 0
        }
    },
    "deployment": {
        "offline": True,
        "network_required_for_inference": False
    }
}

INT8_PATH = f"{EXPORT_DIR}/risk_model_int8.json"

with open(INT8_PATH, "w", encoding="utf-8") as f:
    json.dump(int8_artifact, f, indent=2)

int8_size = os.path.getsize(INT8_PATH)

print("Saved:", INT8_PATH)
print("Size:", int8_size, "bytes")


Saved: ../models/offline/risk_model_int8.json
Size: 1222 bytes


## 10. Local rule engine

The rule engine is deterministic and runs locally.

`lethal_weapon` is optional because it is not one of the five standard interview inputs. If unavailable, the caller supplies `0`.


In [11]:
def local_rule_engine(
    safe_now,
    perpetrator_present,
    can_leave_safely,
    medical_help,
    contact_requested,
    lethal_weapon=0
):
    immediate_threat = (
        int(safe_now) == 0 and
        int(perpetrator_present) == 1
    )

    if immediate_threat and int(can_leave_safely) == 0:
        return "HIGH", "CRITICAL_CURRENT_DANGER"

    if immediate_threat and int(medical_help) == 1:
        return "HIGH", "MEDICAL_DANGER"

    if immediate_threat and int(lethal_weapon) == 1:
        return "HIGH", "LETHAL_WEAPON"

    return None, None


def offline_risk_assessment(
    safe_now,
    perpetrator_present,
    can_leave_safely,
    medical_help,
    contact_requested,
    lethal_weapon=0
):
    x = np.array([[
        safe_now,
        perpetrator_present,
        can_leave_safely,
        medical_help,
        contact_requested
    ]], dtype=np.float64)

    probabilities = standalone_predict_proba(x)[0]

    ml_index = int(np.argmax(probabilities))
    ml_risk = classes[ml_index]
    confidence = float(probabilities[ml_index])

    override, reason = local_rule_engine(
        safe_now,
        perpetrator_present,
        can_leave_safely,
        medical_help,
        contact_requested,
        lethal_weapon
    )

    if override is not None:
        final_risk = override
        source = "RULE_OVERRIDE"
    else:
        final_risk = ml_risk
        source = "ML"

    return {
        "probabilities": {
            classes[i]: float(probabilities[i])
            for i in range(len(classes))
        },
        "ml_risk": ml_risk,
        "confidence": confidence,
        "final_risk": final_risk,
        "decision_source": source,
        "reason": reason
    }


## 11. Test representative offline scenarios

In [12]:
scenarios = [
    {
        "scenario": "Safe",
        "safe_now": 1,
        "perpetrator_present": 0,
        "can_leave_safely": 1,
        "medical_help": 0,
        "contact_requested": 0,
        "lethal_weapon": 0
    },
    {
        "scenario": "Immediate danger + cannot leave",
        "safe_now": 0,
        "perpetrator_present": 1,
        "can_leave_safely": 0,
        "medical_help": 0,
        "contact_requested": 1,
        "lethal_weapon": 0
    },
    {
        "scenario": "Immediate medical danger",
        "safe_now": 0,
        "perpetrator_present": 1,
        "can_leave_safely": 1,
        "medical_help": 1,
        "contact_requested": 1,
        "lethal_weapon": 0
    },
    {
        "scenario": "Weapon + immediate threat",
        "safe_now": 0,
        "perpetrator_present": 1,
        "can_leave_safely": 1,
        "medical_help": 0,
        "contact_requested": 1,
        "lethal_weapon": 1
    }
]

results = []

for scenario in scenarios:
    inputs = {
        k: v
        for k, v in scenario.items()
        if k != "scenario"
    }

    result = offline_risk_assessment(**inputs)

    results.append({
        "scenario": scenario["scenario"],
        "ml_risk": result["ml_risk"],
        "confidence": result["confidence"],
        "final_risk": result["final_risk"],
        "decision_source": result["decision_source"],
        "reason": result["reason"]
    })

display(pd.DataFrame(results))


,scenario,ml_risk,confidence,final_risk,decision_source,reason
0,Safe,LOW,0.998978,LOW,ML,None
1,Immediate danger + cannot leave,HIGH,0.998557,HIGH,RULE_OVERRIDE,CRITICAL_CURRENT_DANGER
2,Immediate medical danger,HIGH,0.983394,HIGH,RULE_OVERRIDE,MEDICAL_DANGER
3,Weapon + immediate threat,MEDIUM,0.806743,HIGH,RULE_OVERRIDE,LETHAL_WEAPON


## 12. Full offline equivalence test

In [13]:
offline_all_pred = standalone_predict(X_all)

print(
    "Offline vs scikit-learn agreement:",
    np.mean(offline_all_pred == sklearn_pred)
)

print(
    "Offline mismatches:",
    np.sum(offline_all_pred != sklearn_pred)
)


Offline vs scikit-learn agreement: 1.0
Offline mismatches: 0


## 13. Model artifact size

In [14]:
print("=" * 70)
print("MODEL SIZE")
print("=" * 70)

print(
    "Original joblib:",
    os.path.getsize(MODEL_PATH),
    "bytes"
)

print(
    "Float JSON:",
    float_size,
    "bytes"
)

print(
    "INT8 JSON:",
    int8_size,
    "bytes"
)

print(
    "INT8 classifier parameters:",
    coefficients_int8.nbytes + intercepts_int8.nbytes,
    "bytes"
)


MODEL SIZE
Original joblib: 2081 bytes
Float JSON: 1365 bytes
INT8 JSON: 1222 bytes
INT8 classifier parameters: 18 bytes


## 14. Local inference latency benchmark

In [15]:
benchmark_input = np.array(
    [1, 0, 1, 0, 0],
    dtype=np.float64
)

N_RUNS = 10000

start = time.perf_counter()

for _ in range(N_RUNS):
    standalone_predict(benchmark_input)

elapsed = time.perf_counter() - start

average_ms = (elapsed / N_RUNS) * 1000

print("Runs:", N_RUNS)
print("Total time:", elapsed, "seconds")
print("Average local inference:", average_ms, "ms")


Runs: 10000
Total time: 0.40382989999852725 seconds
Average local inference: 0.04038298999985273 ms


## 15. Deployment manifest

In [16]:
deployment_manifest = {
    "model": "risk_model_float.json",
    "optional_quantized_model": "risk_model_int8.json",
    "features": MODEL_FEATURES,
    "classes": classes,
    "offline_inference": True,
    "network_required_for_inference": False,
    "rule_engine": "local",
    "input_type": "binary",
    "input_order": MODEL_FEATURES,
    "architecture": [
        "five-question safety interview",
        "local logistic regression",
        "deterministic safety-rule override",
        "LOW/MEDIUM/HIGH output"
    ]
}

MANIFEST_PATH = (
    f"{EXPORT_DIR}/deployment_manifest.json"
)

with open(MANIFEST_PATH, "w", encoding="utf-8") as f:
    json.dump(deployment_manifest, f, indent=2)

print("Saved:", MANIFEST_PATH)


Saved: ../models/offline/deployment_manifest.json


## 16. Final validation

In [17]:
print("=" * 70)
print("FINAL OFFLINE VALIDATION")
print("=" * 70)

print(
    "Float standalone agreement:",
    np.mean(standalone_pred == sklearn_pred)
)

print(
    "INT8 agreement:",
    np.mean(int8_pred == sklearn_pred)
)

print(
    "Float artifact exists:",
    os.path.exists(FLOAT_PATH)
)

print(
    "INT8 artifact exists:",
    os.path.exists(INT8_PATH)
)

print(
    "Rule specification exists:",
    os.path.exists(RULE_PATH)
)

print(
    "Deployment manifest exists:",
    os.path.exists(MANIFEST_PATH)
)

print(
    "Average local inference (ms):",
    average_ms
)


FINAL OFFLINE VALIDATION
Float standalone agreement: 1.0
INT8 agreement: 1.0
Float artifact exists: True
INT8 artifact exists: True
Rule specification exists: True
Deployment manifest exists: True
Average local inference (ms): 0.04038298999985273


# Frozen output

Notebook 06 produces:

```text
models/offline/
├── risk_model_float.json
├── risk_model_int8.json
└── deployment_manifest.json
```

The original trained artifacts remain:

```text
models/
├── risk_logistic_regression.joblib
├── risk_rule_engine.json
└── risk_model_metadata.json
```

Final inference:

**5 safety answers → local Logistic Regression → local rule engine → LOW/MEDIUM/HIGH**

No network call is required for risk inference.
